<a href="https://colab.research.google.com/github/lu-damia-ru/bat-shell/blob/master/iDanceDownloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Pulizia totale di vecchi driver e browser conflittuali
!apt-get remove google-chrome-stable chromium-browser chromium-chromedriver -y
!rm -f /usr/bin/chromedriver

# 2. Aggiornamento repository e installazione delle dipendenze di sistema indispensabili per Chrome Headless
!apt-get update
!apt-get install -y libxss1 libappindicator1 libgconf-2-4 libxi6 libgbm1 fonts-liberation libasound2

# 3. Installa l'ultima versione stabile di Google Chrome per Linux
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb

# 4. Pulizia file temporaneo .deb
!rm -f google-chrome-stable_current_amd64.deb

# 5. Installa le librerie Python aggiornate (Selenium >= 4.11 gestisce il driver da solo!)
!pip install selenium beautifulsoup4 requests -U

# 6. Verifica rapida delle versioni installate
!echo "=== VERIFICA COMPONENTI ==="
!google-chrome --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package google-chrome-stable
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:12 http://security.ubuntu.com/ubunt

In [5]:
import os
from google.colab import drive

# --- COLLEGAMENTO GOOGLE DRIVE ---
print("[LOG] Collegamento a Google Drive...")
drive.mount('/content/drive')

# --- CONFIGURAZIONE ACCOUNT IDANCE ---
IDANCE_EMAIL = "luigidamiano.russo@gmail.com"
IDANCE_PASSWORD = "Luigi21idance_"

# --- CONFIGURAZIONE PERCORSI GLOBALI GENERICI ---
BASE_DIR = "/content/drive/My Drive/Colab Notebooks/buffer_test"
os.makedirs(BASE_DIR, exist_ok=True)

# Il file unico di interscambio e tracciamento per l'uploader di YouTube
NOME_FILE_CSV = os.path.join(BASE_DIR, "ytuploadtest.csv")

# --- PLAYLIST YOUTUBE PREDEFINITA ---
URL_PLAYLIST_UNICA = "https://www.youtube.com/playlist?list=PL9u4YZN184361VfS0GCdqXeXTyQqp5yYn"

# Elenco delle colonne fondamentali (incluse le chiavi di sincronizzazione per YouTube)
COLONNE_CSV = ["nome_file_originale", "titolo_youtube", "descrizione_youtube", "playlist_url", "youtube_id"]

# Elenco degli indici di iDance da scansionare (es: Milonga, Salsa, Bachata, ecc.)
URLS_DA_SCANSIONARE = [
    "https://www.idance.net/en/lessons/search?search%5Bstyle%5D=West+Coast+Swing"
]


# Inizializza il file CSV con l'intestazione standard se non esiste già
if not os.path.exists(NOME_FILE_CSV):
    with open(NOME_FILE_CSV, "w", encoding="utf-8", newline="") as f:
        import csv
        writer = csv.DictWriter(f, fieldnames=COLONNE_CSV, delimiter=";")
        writer.writeheader()
    print("[SISTEMA] Creato nuovo file CSV di tracciamento standard: ytupload.csv")
else:
    print("[SISTEMA] File CSV di tracciamento rilevato e agganciato.")

[LOG] Collegamento a Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SISTEMA] File CSV di tracciamento rilevato e agganciato.


In [6]:
import os
import re
import csv
import time
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- CONFIGURAZIONE AMBIENTE SELENIUM HEADLESS ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=chrome_options)


def normalizza_titolo_youtube(titolo_grezzo, indice_scansione=1):
    """
    ENGINE EURISTICO AVANZATO
    Analizza il titolo con logica a cascata per garantire SEMPRE un progressivo numerico in testa.
    """
    # Rimuove spazi doppi e pulisce caratteri strani iniziali/finali
    titolo_grezzo = " ".join(titolo_grezzo.split()).strip()

    # Pre-pulizia dai caratteri vietati nei file system per evitare crash di scrittura su disco
    titolo_pulito_base = re.sub(r'[\\/*?:"<>|]', '', titolo_grezzo)

    # ------------------------------------------------------------------------
    # CASO 1: Formato classico Tango (Es: Argentine Tango Vol 2, Lesson 5...)
    # ------------------------------------------------------------------------
    match_tango = re.search(r'(?:Argentine\s+Tango\s+)?Vol(?:ume)?\s+(\d+),\s+(?:Lesson|Lezione)\s+(\d+)', titolo_grezzo, re.IGNORECASE)
    if match_tango:
        vol = match_tango.group(1)
        lesson = int(match_tango.group(2))
        lesson_pad = f"{lesson:02d}"  # Trasforma '5' in '05' per l'ordinamento alfabetico corretto

        # Rimuove la stringa del volume/lezione dalla coda del titolo originale per non duplicarla
        titolo_snello = re.sub(r'\s*-\s*(?:Argentine\s+Tango\s+)?Vol(?:ume)?\s+\d+,\s+(?:Lesson|Lezione)\s+\d+', '', titolo_pulito_base, flags=re.IGNORECASE).strip()
        titolo_snello = titolo_snello.rstrip('-').strip()

        return f"V{vol}.L{lesson_pad} - {titolo_snello}", f"Vol {vol}", f"Lezione {lesson_pad}"

    # ------------------------------------------------------------------------
    # CASO 2: Formato frazionario generico di altri insegnanti (Es: "12 Of 22")
    # ------------------------------------------------------------------------
    match_frazione = re.search(r'(\d+)\s+o[ff]\s+(\d+)', titolo_grezzo, re.IGNORECASE)
    if match_frazione:
        corrente = int(match_frazione.group(1))
        totale = match_frazione.group(2)
        corrente_pad = f"{corrente:03d}"  # 3 cifre (es: 012) copre serie molto lunghe

        # Pulisce il titolo rimuovendo la parte "X of N"
        titolo_snello = re.sub(r',?\s*\d+\s+o[ff]\s+\d+', '', titolo_pulito_base, flags=re.IGNORECASE).strip()
        titolo_snello = titolo_snello.strip(',- ').strip()

        return f"{corrente_pad} - {titolo_snello}", "Generico", f"{corrente_pad} di {totale}"

    # ------------------------------------------------------------------------
    # CASO 3: Numero progressivo singolo o parziale (Es: "Lesson 4", "Part 02", "#15")
    # ------------------------------------------------------------------------
    match_isolato = re.search(r'(?:Lesson|Lezione|Part|Parte|#)\s*(\d+)', titolo_grezzo, re.IGNORECASE)
    if match_isolato:
        num = int(match_isolato.group(1))
        num_pad = f"{num:03d}"
        return f"{num_pad} - {titolo_pulito_base}", "Generico", f"Prog {num_pad}"

    # ------------------------------------------------------------------------
    # CASO 4: Caccia ai numeri nudi (Cerca se c'è un qualsiasi numero isolato nel testo)
    # ------------------------------------------------------------------------
    match_nudo = re.search(r'\b(\d+)\b', titolo_grezzo)
    if match_nudo:
        num = int(match_nudo.group(1))
        num_pad = f"{num:03d}"
        return f"{num_pad} - {titolo_pulito_base}", "Generico", f"Prog {num_pad}"

    # ------------------------------------------------------------------------
    # CASO 5: Fallback Totale (Nessun numero o progressivo trovato nel titolo)
    # ------------------------------------------------------------------------
    # Se il titolo è puramente testuale, usa la posizione cronologica del link nella pagina
    auto_index = f"{indice_scansione:03d}"
    return f"[AUTO_{auto_index}] - {titolo_pulito_base}", "Sconosciuto", f"Auto {auto_index}"


def esegui_login():
    """Effettua il login sul portale iDance usando Selenium."""
    print("[LOG] Connessione alla Home Page di iDance...")
    driver.get("https://www.idance.net/")
    time.sleep(5)

    try:
        print("[LOG] Clicco sul link 'Sign In | Sign Up'...")
        link_pop_up = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "open_sign_in_or_register_modal"))
        )
        link_pop_up.click()
        time.sleep(4)

        print("[LOG] Inserisco le credenziali nel pop-up...")
        campo_email = driver.find_element(By.ID, "sign_in_user_email")
        campo_pass = driver.find_element(By.ID, "sign_in_user_password")

        campo_email.clear()
        campo_email.send_keys(IDANCE_EMAIL)
        campo_pass.clear()
        campo_pass.send_keys(IDANCE_PASSWORD)

        try:
            bottone_login = driver.find_element(By.XPATH, "//form//input[@type='submit'] | //form//button[@type='submit']")
        except:
            bottone_login = driver.find_element(By.NAME, "commit")

        bottone_login.click()
        print("[LOG] Login inviato, attendo caricamento sessione...")
        time.sleep(6)

        print("[OK] Login effettuato con successo!")
        return True
    except Exception as e:
        print(f"[ERRORE CRITICO] Flusso di login fallito: {e}")
        return False


def analizza_e_scarica_lezione(url_lezione, nome_file_pulito, percorso_video_drive):
    """Analizza la pagina interna della lezione ed effettua lo stream del download direttamente nel buffer."""
    print(f"\n[NAVIGAZIONE] Entro in: {url_lezione}")
    driver.get(url_lezione)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    testo_completo = soup.get_text(" ", strip=True)

    insegnanti = "Insegnanti non trovati"
    if "Instructors:" in testo_completo:
        insegnanti = testo_completo.split("Instructors:")[1].split("Level:")[0].strip()

    descrizione_real = "Descrizione non trovata"
    if "Lesson Details" in testo_completo:
        descrizione_real = testo_completo.split("Lesson Details")[1].split("Genre:")[0].strip()

    url_video_reale = None
    video_tag = soup.find('video', class_='lesson_full')
    if video_tag:
        source_tag = video_tag.find('source')
        if source_tag and source_tag.get('src'):
            url_video_reale = source_tag['src']
            url_video_reale = urljoin(url_lezione, url_video_reale)
            print("  [SISTEMA] URL video intero catturato dal codice sorgente!")
    else:
        print("  [AVVISO] Impossibile trovare il video 'lesson_full'. Salto.")
        return insegnanti, descrizione_real, False

    if url_video_reale:
        print(f"  [DOWNLOAD] Scarico il file direttamente nel buffer di Drive...")

        cookies_list = driver.get_cookies()
        session_requests = requests.Session()
        for cookie in cookies_list:
            session_requests.cookies.set(cookie['name'], cookie['value'])

        r = session_requests.get(url_video_reale, stream=True)
        if r.status_code == 200:
            with open(percorso_video_drive, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
            print("  [OK] Video scaricato con successo.")
            return insegnanti, descrizione_real, True
        else:
            print(f"  [ERRORE] Errore di download dal server multimediale. Status: {r.status_code}")

    return insegnanti, descrizione_real, False


def recupera_titoli_gia_a_catalogo():
    """Legge il file CSV per saltare i record già scritti ed evitare duplicati."""
    gia_presenti = set()
    if os.path.exists(NOME_FILE_CSV):
        with open(NOME_FILE_CSV, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=";")
            for row in reader:
                if row.get('nome_file_originale'):
                    gia_presenti.add(row['nome_file_originale'])
    return gia_presenti


def main():
    if not esegui_login():
        return

    # Recupera l'elenco dei video storici salvati nel CSV generico di interscambio
    video_gia_a_tabella = recupera_titoli_gia_a_catalogo()
    print(f"[SISTEMA] Rilevati {len(video_gia_a_tabella)} elementi nel database CSV.")

    for url_ricerca in URLS_DA_SCANSIONARE:
        print(f"\n[LISTA] Lettura indice principale: {url_ricerca}")
        driver.get(url_ricerca)
        time.sleep(5)

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        link_lezioni = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if "/lessons/" in href and "search" not in href:
                titolo_testo = a.get_text(strip=True)
                if len(titolo_testo) > 3 and not any(href == e[0] for e in link_lezioni):
                    link_lezioni.append((href, titolo_testo))

        print(f"[LOG] Trovate {len(link_lezioni)} lezioni potenziali in questa pagina.")

        # ====================================================================
        # INIZIO DEL CICLO MODIFICATO CON ENUMERATE ED EURISTICA
        # ====================================================================
        for idx, (href, titolo_principale) in enumerate(link_lezioni, start=1):
            url_completo = urljoin(url_ricerca, href)

            # Passiamo l'indice numerico progressivo (idx) all'engine di normalizzazione euristica
            titolo_definitivo_yt, vol, lesson = normalizza_titolo_youtube(titolo_principale, indice_scansione=idx)

            nome_file_completo = f"{titolo_definitivo_yt}.mp4"
            percorso_video_drive = os.path.join(BASE_DIR, nome_file_completo)

            # --- CONTROLLO RIPRESA AUTOMATICA (CSV) ---
            if nome_file_completo in video_gia_a_tabella:
                print(f"[SKIP] Il video '{nome_file_completo}' è già presente nel CSV. Lo salto.")
                continue

            # --- CONTROLLO FISICO FILE SYSTEM ---
            if os.path.exists(percorso_video_drive):
                print(f"[SKIP CLONE] File fisico presente nel buffer ma assente nel CSV. Genero i metadati.")
                insegnanti, descrizione_real, scaricato = analizza_e_scarica_lezione(url_completo, titolo_definitivo_yt, percorso_video_drive)
                scaricato = True  # Forza la scrittura per riallineare il CSV al disco
            else:
                # Esegue la procedura standard di analisi della pagina e download effettivo
                insegnanti, descrizione_real, scaricato = analizza_e_scarica_lezione(url_completo, titolo_definitivo_yt, percorso_video_drive)

            if scaricato:
                descrizione_finale_yt = f"Insegnanti: {insegnanti}\nVolume: {vol} - Lezione: {lesson}\n\n{descrizione_real}"

                # SCRITTURA ATOMICA IMMEDIATA IN APPEND SUL CSV
                riga_dati = {
                    "nome_file_originale": nome_file_completo,
                    "titolo_youtube": titolo_definitivo_yt,
                    "descrizione_youtube": descrizione_finale_yt,
                    "playlist_url": URL_PLAYLIST_UNICA,
                    "youtube_id": ""  # Campo vuoto che verrà valorizzato dal secondo notebook (Uploader)
                }

                with open(NOME_FILE_CSV, "a", encoding="utf-8", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=COLONNE_CSV, delimiter=";")
                    writer.writerow(riga_dati)

                # Memorizza nel tracciamento locale per evitare duplicati istantanei nello stesso loop
                video_gia_a_tabella.add(nome_file_completo)
                print(f"  [CSV] Registrato nel database: {titolo_definitivo_yt}")
            else:
                print(f"  [AVVISO] Registrazione CSV saltata per mancato scaricamento di: {titolo_principale}")

            time.sleep(3)

    print(f"\n=== PROCESSO COMPLETATO ===")
    print(f"Tutti i video estratti e normalizzati si trovano in: '{BASE_DIR}'")
    driver.quit()

if __name__ == "__main__":
    main()

[LOG] Connessione alla Home Page di iDance...
[LOG] Clicco sul link 'Sign In | Sign Up'...
[LOG] Inserisco le credenziali nel pop-up...
[LOG] Login inviato, attendo caricamento sessione...
[OK] Login effettuato con successo!
[SISTEMA] Rilevati 0 elementi nel database CSV.

[LISTA] Lettura indice principale: https://www.idance.net/en/lessons/search?search%5Bstyle%5D=West+Coast+Swing
[LOG] Trovate 30 lezioni potenziali in questa pagina.

[NAVIGAZIONE] Entro in: https://www.idance.net/en/lessons/2794-all-22-patterns-in-context-west-coast-swing-foundation-patterns-online-west-coast-swing-dance-lesson-with-skippy-blair
  [SISTEMA] URL video intero catturato dal codice sorgente!
  [DOWNLOAD] Scarico il file direttamente nel buffer di Drive...
  [OK] Video scaricato con successo.
  [CSV] Registrato nel database: 022 - All 22 Patterns In Context - West Coast Swing Foundation Patterns

[NAVIGAZIONE] Entro in: https://www.idance.net/en/lessons/2789-release-whip-west-coast-swing-foundation-patter